In [1]:
%load_ext autoreload
import datetime as dt
import os
import sys
from itertools import product

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, project_root)

In [2]:
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from ipywidgets import interact
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf

from src.data.download_yfinance import download_ticker_data

### Load data

In [3]:
# declare period to load into notebook 
# Note: (yfinance will have freshest data every morning CEST)
params = {
    'start_date': '2022-01-01',
    'end_date': dt.datetime.today()
}

In [4]:
# declare tickers + tickers sanity test
with open("../config/tickers_list.yaml", "r") as f:
    tickers = yaml.safe_load(f)["tickers"]

tickers[:5]

['AAPL', 'MSFT', 'GDX', 'IONQ', '7974.T']

In [5]:
intervals = ["1d", "1wk", "1mo"]
all_dfs = []

for ticker, interval in product(tickers, intervals):
    df_i = download_ticker_data(
        tickers=[ticker],
        start=params["start_date"],
        end=params["end_date"],
        interval=interval,
    )
    if df_i is not None and not df_i.empty:
        df_i["ticker"] = ticker
        df_i["interval"] = interval
        all_dfs.append(df_i)

df_all = pd.concat(all_dfs)

### Basic checks

In [6]:
print(f'Number of rows and columns: {df_all.shape[0], df_all.shape[1]}')

Number of rows and columns: (34695, 8)


In [7]:
print(f'data starts: {df_all.date.min()}')
print(f'data ends: end_date = {df_all.date.max()}')

data starts: 2022-01-01 00:00:00
data ends: end_date = 2025-06-30 00:00:00


### Plot options

In [8]:
ticker = widgets.Dropdown(options=sorted(tickers), description="Ticker")
interval = widgets.RadioButtons(options=["1d", "1wk", "1mo"])
seasonality_focus = widgets.RadioButtons(
    options=["generic", "yearly"], 
    description="Seasonality")

In [ ]:
# TODO: create fake ticker with perfect seasonality for comparison.
# it should be added to the tickers dataset

# Dashboards
## STL Seasonality Strength

Quantifies how much of a time series’ variation is explained by its seasonal component, as extracted using STL (Seasonal-Trend decomposition using Loess). It is calculated as:

$$S = \max\left(0,\ 1 - \frac{\operatorname{Var}(\text{remainder})}{\operatorname{Var}(\text{remainder} + \text{seasonal})} \right)$$


Values near 1 indicate strong seasonality; near 0 means weak or no seasonality.

In [9]:
@interact(ticker=ticker, interval=interval, seasonality=seasonality_focus)
def show_stl_strength(ticker, interval, seasonality):
    df = df_all[
        (df_all["ticker"] == ticker) & (df_all["interval"] == interval)
        ].copy()
    df["date"] = pd.to_datetime(df["date"])
    
    if df.empty:
        print("No data available.")
        return

    df.set_index("date", inplace=True)
    df = df.sort_index()
    series = df["close"].dropna()

    seasonal_periods = {
        "1d": {"generic": 252, "yearly": 365},
        "1wk": {"generic": 52, "yearly": 52},
        "1mo": {"generic": 12, "yearly": 12}
    }
    period = seasonal_periods[interval][seasonality]

    try:
        stl = STL(series, period=period, robust=True)
        result = stl.fit()
    except Exception as e:
        print(f"STL failed: {e}")
        return

    remainder = result.resid
    seasonal = result.seasonal
    strength = max(0, 1 - (np.var(remainder) / np.var(remainder + seasonal)))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=series.index, 
        y=series, 
        name="Close"))
    fig.add_trace(go.Scatter(
        x=series.index, 
        y=result.trend, 
        name="Trend"))
    fig.add_trace(go.Scatter(
        x=series.index, 
        y=seasonal, 
        name="Seasonal"))

    fig.update_layout(
        title= (
            f"{ticker} ({interval}) — STL Seasonality Strength"
            f"({seasonality}): {strength:.2f}"
        ),
        xaxis_title="Date",
        yaxis_title="Price",
        template="plotly_white"
    )
    fig.show()

interactive(children=(Dropdown(description='Ticker', options=('7974.T', 'AAPL', 'AVAV', 'BB', 'BMBL', 'CGW', '…

## ACF at Seasonal Lag

Refers to the value of the autocorrelation function at the lag corresponding to the seasonality period (e.g., lag = 12 for monthly data with yearly seasonality).

$$\text{ACF}_{s} = \rho(s)$$

where:
- $\text{ACF}_{s}$ is the autocorrelation at seasonal lag $s$
- $\rho(s)$ denotes the autocorrelation function evaluated at lag $s$

In [10]:
@interact(ticker=ticker, interval=interval)
def show_acf_seasonality(ticker, interval):
    df = df_all[(df_all["ticker"] == ticker) & (df_all["interval"] == interval)].copy()
    df["date"] = pd.to_datetime(df["date"])
    
    if df.empty:
        print("No data available.")
        return

    df.set_index("date", inplace=True)
    df = df.sort_index()
    series = df["close"].dropna()

    # seasonal lag definitions
    seasonal_lags = {
        "1d": 252,
        "1wk": 52,
        "1mo": 12
    }
    lag = seasonal_lags.get(interval, 12)

    # compute ACF
    max_lag = lag + 10
    acf_vals = acf(series, nlags=max_lag, fft=True)

    # value at seasonal lag
    seasonal_acf = acf_vals[lag]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=list(range(len(acf_vals))), 
        y=acf_vals, 
        name="ACF"))
    fig.add_trace(go.Scatter(
        x=[lag], 
        y=[seasonal_acf], 
        mode="markers+text", 
        text=[f"{seasonal_acf:.2f}"], 
        textposition="top center",
        marker=dict(color="red", size=10),
        name=f"ACF @ lag {lag}"
    ))

    fig.update_layout(
        title=f"{ticker} ({interval}) — ACF at Lag {lag}: {seasonal_acf:.2f}",
        xaxis_title="Lag",
        yaxis_title="ACF",
        template="plotly_white"
    )
    fig.show()

interactive(children=(Dropdown(description='Ticker', options=('7974.T', 'AAPL', 'AVAV', 'BB', 'BMBL', 'CGW', '…